# Fine-tune, prompt or retrieve?

Machine Learning Engineer track, lesson MLE11. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0. The questions are about the site's own SQL lessons, also CC BY 4.0.

**Goals:** try one task three ways on the same open model (prompt it, give it retrieved text, tune it with LoRA), read the configuration, the loss curve and the evaluation of one real LoRA run, and decide which of the three a stated problem needs.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. No key and no account are needed: both models are open and run in this notebook. A free CPU runtime is enough (the language model is about 3.3 GB to download; the cells that ask it many questions and the training cell take several minutes each on a CPU); a GPU runtime may be faster, but it was not timed.

## Setup

In [ ]:
import importlib.metadata
import json
import math
import numbers
import os
import re
import statistics
import subprocess
import sys
import tempfile
import time
import urllib.request
from typing import Literal

for pin in ["transformers==5.24.1", "sentence-transformers==6.3.0", "pydantic==2.14.4", "peft==0.22.0"]:                                                         # the versions this notebook was run with
    package, version = pin.split("==")
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:                                                  # install the pinned version when Colab has another
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pin], check=True)

import numpy as np
import torch
from pydantic import BaseModel, ValidationError
from peft import LoraConfig, get_peft_model
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 2026       # the seed of the whole track
torch.set_num_threads(2)       # two threads, the size of a free Colab CPU

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## The corpus, the two models and the questions

The corpus is the one of lesson MLE9: eight lessons of the SQL stage, from the release tag `v1.0.0`. The two models are the small encoder `all-MiniLM-L6-v2` and the open model `granite-4.0-1b` (made by IBM, Apache 2.0, downloaded without an account), each pinned to one revision, with the `chat` function of lesson MLE8. Run these cells as they are.

In [ ]:
# The corpus: eight published lessons of the SQL stage (CC BY 4.0), read from the release tag v1.0.0, or from this repository's own copy of the same files.
TEXT_BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v1.0.0/src/content/docs/sql/"
TEXT_LOCAL = "../../src/content/docs/sql/"      # the same files, when this notebook runs inside a clone of the repository
LESSONS = ["null-handling", "group-by-having", "inner-join", "left-join-anti-join", "window-functions", "ctes", "dates-and-time", "case-expressions"]
SKIP_SECTIONS = {"Warm-up", "Finish it", "On your own", "Check yourself"}      # recall prompts and exercises are not teaching text


def read_lesson(slug):
    """The raw text of one lesson file: downloaded from the tag, or the local copy."""
    try:
        with urllib.request.urlopen(TEXT_BASE + slug + ".mdx", timeout=30) as reply:
            return reply.read().decode("utf-8")
    except Exception:
        with open(TEXT_LOCAL + slug + ".mdx", encoding="utf-8") as handle:
            return handle.read()


def paragraphs(raw):
    """The prose paragraphs of a lesson as (section heading, text): no front matter, figures, code, tables or exercise sections."""
    raw = re.sub(r"\A---\n.*?\n---\n", "", raw, flags=re.S)
    raw = re.sub(r"<figure.*?</figure>|<svg.*?</svg>|<RunSQL.*?/>|```.*?```", "", raw, flags=re.S)
    found, section = [], ""
    for block in re.split(r"\n\s*\n", raw):
        block = block.strip()
        if block.startswith("#"):
            section = block.lstrip("#").strip()
        elif block and section not in SKIP_SECTIONS and not block.startswith(("<", "|", "{/*")):
            block = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", re.sub(r"<[^>]+>", "", block)).replace("*", "").replace("`", "")
            block = re.sub(r"\s+", " ", block).strip()
            if len(block.split()) >= 8:                                           # a few words are a label, not a paragraph
                found.append((section, block))
    return found


CORPUS = {slug: paragraphs(read_lesson(slug)) for slug in LESSONS}
paras = [(slug, text) for slug in LESSONS for _, text in CORPUS[slug]]            # (lesson, paragraph) for every paragraph
texts, labels = [text for _, text in paras], [slug for slug, _ in paras]
print(len(paras), "paragraphs,", sum(len(text.split()) for text in texts), "words,", len(LESSONS), "lessons")

In [ ]:
EMB_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMB_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"                    # the exact version of the model files
emb = SentenceTransformer(EMB_NAME, revision=EMB_REVISION, device="cpu")
print(sum(p.numel() for p in emb.parameters()), emb.max_seq_length)

In [ ]:
LLM_NAME = "ibm-granite/granite-4.0-1b"
LLM_REVISION = "6a7381ba1f54d684ff508d991aeb7dc580157103"                    # the exact version of the model files
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(LLM_NAME, revision=LLM_REVISION)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, revision=LLM_REVISION, dtype=torch.float32).to(DEVICE).eval()
END_MARKS = {tok.eos_token_id}                                              # the end-of-text marker: <|end_of_text|>


def chat(messages, max_new_tokens=100, temperature=0.0, seed=None):
    """One request. temperature 0 picks the most likely token every time; above 0 it samples."""
    prompt = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"].to(DEVICE)
    settings = {"do_sample": False, "repetition_penalty": 1.0}
    if temperature > 0:
        settings = {"do_sample": True, "temperature": temperature, "top_p": 1.0, "top_k": 0, "repetition_penalty": 1.0}
        torch.manual_seed(SEED if seed is None else seed)
    start = time.perf_counter()
    with torch.no_grad():
        out = llm.generate(input_ids=prompt, attention_mask=torch.ones_like(prompt), max_new_tokens=max_new_tokens, **settings)
    seconds = time.perf_counter() - start
    written = out[0][prompt.shape[1]:].tolist()
    return {"text": tok.decode(written, skip_special_tokens=True), "prompt_tokens": prompt.shape[1],
            "completion_tokens": sum(1 for t in written if t not in END_MARKS), "seconds": seconds}


print(sum(p.numel() for p in llm.parameters()), DEVICE)

The task is the routing of lesson MLE8: read a learner's question and name the lesson of the eight that answers it. `TRAIN` is 96 questions written for this lesson, 12 for each lesson, and the only questions the model is trained on. `DEV` is the 16 questions of lesson MLE9, used to watch the loss while training. `TEST` is the 16 golden questions of lesson MLE10 and 16 new ones, 2 for each lesson: scored once for each approach and never used to choose anything. Run this cell as it is.

In [ ]:
TRAIN = [
    ('Which test finds the rows where a phone number is missing?', 'null-handling'),
    ('Is a missing price the same as a price of zero?', 'null-handling'),
    ('What does 5 + NULL give?', 'null-handling'),
    ('Why is my COUNT of a column lower than the number of rows?', 'null-handling'),
    ('What does NULL = NULL return?', 'null-handling'),
    ('How do I show the text unknown where a value is missing?', 'null-handling'),
    ('Which is safer than NOT IN when a subquery can return empty values?', 'null-handling'),
    ('How do I put a 0 into empty quantities in a report?', 'null-handling'),
    ('How can I keep only the rows where a discount is not missing?', 'null-handling'),
    ('What does a comparison with an empty value evaluate to?', 'null-handling'),
    ('How do I replace an empty store name with online?', 'null-handling'),
    ('Why does WHERE age <> 30 drop the rows with no age?', 'null-handling'),
    ('How do I count the orders of each store?', 'group-by-having'),
    ('How do I get the total sales of every product?', 'group-by-having'),
    ('Which clause keeps only the stores with more than 100 orders?', 'group-by-having'),
    ('What does the grain of a query mean?', 'group-by-having'),
    ('How do I group by store and by month at once?', 'group-by-having'),
    ('Why does the database say a column must appear in the GROUP BY?', 'group-by-having'),
    ('How can I check that the group counts add up to the total?', 'group-by-having'),
    ('How do I average the order total of each customer?', 'group-by-having'),
    ('Do rows removed by a filter still count in a group?', 'group-by-having'),
    ('How many rows does grouping by store return when there are 3 stores?', 'group-by-having'),
    ('How do I find the products that sold fewer than 50 units?', 'group-by-having'),
    ('Why do I get one row per store and product instead of one per store?', 'group-by-having'),
    ('How do I combine orders with stores on store_id?', 'inner-join'),
    ('How do I add the store name to every order row?', 'inner-join'),
    ('Why did my join return fewer rows than the orders table?', 'inner-join'),
    ('Why did the row count go up after my join?', 'inner-join'),
    ('How do I join order lines to products to see product names?', 'inner-join'),
    ('What is a join key?', 'inner-join'),
    ('How do I select columns from both tables after a join?', 'inner-join'),
    ('How do I check that a join did not lose or duplicate rows?', 'inner-join'),
    ('Why did one order show up three times after the join?', 'inner-join'),
    ('How do I pair each order line with its order date?', 'inner-join'),
    ('Does the order of the two tables matter in an inner join?', 'inner-join'),
    ('How do I give a table a short alias in a join?', 'inner-join'),
    ('How do I keep the stores that have no orders in my result?', 'left-join-anti-join'),
    ('How do I list the products that were never sold?', 'left-join-anti-join'),
    ('Which join returns all customers, even the ones without orders?', 'left-join-anti-join'),
    ('How do I find orders that have no matching order lines?', 'left-join-anti-join'),
    ('Should a filter on the right table go in ON or in WHERE?', 'left-join-anti-join'),
    ('How do I count the customers who never ordered?', 'left-join-anti-join'),
    ('What is an anti-join?', 'left-join-anti-join'),
    ('What is the difference between LEFT JOIN and FULL JOIN?', 'left-join-anti-join'),
    ('How do I keep the rows of both tables whether or not they match?', 'left-join-anti-join'),
    ('Why does my left join return the same rows as an inner join?', 'left-join-anti-join'),
    ('How do I use NOT EXISTS to find rows with no partner?', 'left-join-anti-join'),
    ("How do I use IS NULL on the right table's key to find the lonely rows?", 'left-join-anti-join'),
    ('How do I add a running total of sales by date?', 'window-functions'),
    ("How do I show each order's share of its day's total?", 'window-functions'),
    ('What does OVER do?', 'window-functions'),
    ('Why do tied rows get the same running total?', 'window-functions'),
    ('How do I restart a running total for every store?', 'window-functions'),
    ('How do I compare each row with the average of its group without collapsing the rows?', 'window-functions'),
    ('Why did the number of rows stay the same after I added SUM OVER?', 'window-functions'),
    ('How do I get a cumulative count of orders over time?', 'window-functions'),
    ('How is a window function different from GROUP BY?', 'window-functions'),
    ("How do I compute each store's share of total sales and keep every store row?", 'window-functions'),
    ('What does ORDER BY inside OVER change?', 'window-functions'),
    ('Which clause fences the rows into blocks without collapsing them?', 'window-functions'),
    ('How do I split a long query into named steps?', 'ctes'),
    ('What does WITH do at the start of a query?', 'ctes'),
    ('How do I reuse the same subquery twice in one query?', 'ctes'),
    ('How do I check the row count of each step of a long query?', 'ctes'),
    ('How do I make a query readable by building it one stage at a time?', 'ctes'),
    ('How do I name an intermediate result that I use later in the same query?', 'ctes'),
    ('Can the second step read the first one?', 'ctes'),
    ('Is a CTE the same as a temporary table?', 'ctes'),
    ('Why use a CTE instead of a nested subquery?', 'ctes'),
    ('How do I average the per-store totals in one query?', 'ctes'),
    ('Can I select from a named step like a table?', 'ctes'),
    ('How do I build a query step by step and check each step?', 'ctes'),
    ('How do I get the year of an order date?', 'dates-and-time'),
    ('How do I find the day of the week of a timestamp?', 'dates-and-time'),
    ('How do I add 7 days to a date?', 'dates-and-time'),
    ('How do I get the number of days between two dates?', 'dates-and-time'),
    ('How do I group orders by week?', 'dates-and-time'),
    ('How do I count orders per month?', 'dates-and-time'),
    ('Why is the last day of the month short in my daily counts?', 'dates-and-time'),
    ('How do I know whether a month is only partly covered by the data?', 'dates-and-time'),
    ('How do I get the hour of a timestamp?', 'dates-and-time'),
    ('Is a date stored as text or as a number of days?', 'dates-and-time'),
    ('What does INTERVAL 1 MONTH do?', 'dates-and-time'),
    ('How do I turn a timestamp into a date without the time?', 'dates-and-time'),
    ('How do I label orders as small, medium or large?', 'case-expressions'),
    ('How do I put sales into buckets by amount?', 'case-expressions'),
    ('Why did my CASE return large for an order that is also medium?', 'case-expressions'),
    ('How do I make a flag column that is 1 for weekend orders and 0 otherwise?', 'case-expressions'),
    ('How do I combine two conditions in one WHEN?', 'case-expressions'),
    ('How do I count how many orders fall in each bucket?', 'case-expressions'),
    ('How do I turn payment method codes into readable names?', 'case-expressions'),
    ('How do I write an IF-THEN-ELSE in SQL?', 'case-expressions'),
    ('How do I mark rows as peak hours?', 'case-expressions'),
    ('What is the SQL version of the Excel IF flag column?', 'case-expressions'),
    ('How do I check that every row got exactly one label?', 'case-expressions'),
    ('How do I bucket customers by their number of orders?', 'case-expressions'),
]
TEST_NEW = [
    ('How can I tell a real zero from a missing value in a column?', 'null-handling'),
    ('How do I keep the rows whose discount is missing in a filter on discount below 5?', 'null-handling'),
    ('How do I list the stores whose average order is above 6 dollars?', 'group-by-having'),
    ('Why must every selected column be a group key or inside an aggregate?', 'group-by-having'),
    ('How do I link order lines to their orders?', 'inner-join'),
    ('Why does my joined result have more rows than either table?', 'inner-join'),
    ('How do I report every product, with zero for the products that never sold?', 'left-join-anti-join'),
    ('How do I keep all stores in a monthly report, even those with no sales?', 'left-join-anti-join'),
    ('How do I add a running count of orders for each customer?', 'window-functions'),
    ("How can I show each product's sales next to its category total?", 'window-functions'),
    ('How do I compute daily totals first and then their average?', 'ctes'),
    ('How can I give a sub-result a name that the rest of the query uses?', 'ctes'),
    ('How many days are there between an order and its delivery date?', 'dates-and-time'),
    ('How do I list the orders placed in the last 30 days of the data?', 'dates-and-time'),
    ('How do I show a tick mark for the orders with a discount?', 'case-expressions'),
    ('How do I turn the number of items into single or multi?', 'case-expressions'),
]
DEV = [
    ('Why does WHERE customer_id = NULL never return a row?', 'null-handling'),
    ('Which function replaces a missing value with a default?', 'null-handling'),
    ('What is the difference between WHERE and HAVING?', 'group-by-having'),
    ('How many orders does store 2 have in all?', 'group-by-having'),
    ('What happens to a row that has no partner in an inner join?', 'inner-join'),
    ('If a key repeats on both sides, how many rows does a join return?', 'inner-join'),
    ('How do I find customers who never placed an order?', 'left-join-anti-join'),
    ('Why did my LEFT JOIN lose the unmatched rows after I added a WHERE on the right table?', 'left-join-anti-join'),
    ('How can I add a total beside every row without losing rows?', 'window-functions'),
    ('Can I filter on the result of a window function in WHERE?', 'window-functions'),
    ('What is a CTE?', 'ctes'),
    ('Does a CTE stay in the database after the query ends?', 'ctes'),
    ('How do I get the first day of the month of a timestamp?', 'dates-and-time'),
    ('Why does BETWEEN miss orders placed on the last day of a month?', 'dates-and-time'),
    ('What does CASE return when no WHEN is true and there is no ELSE?', 'case-expressions'),
    ('Which WHEN wins when two tests are both true?', 'case-expressions'),
]
GOLD = [
    ('Why does a NOT IN list that holds a NULL return no rows?', 'null-handling'),
    ('Do aggregate functions such as AVG count missing values?', 'null-handling'),
    ('In which order does SQL work through the clauses of a query?', 'group-by-having'),
    ('What does every group become after GROUP BY?', 'group-by-having'),
    ('What does the ON condition of a join name?', 'inner-join'),
    ('Is JOIN on its own the same as INNER JOIN?', 'inner-join'),
    ("What goes in the right table's columns when a row of a LEFT JOIN finds no match?", 'left-join-anti-join'),
    ('What does a RIGHT JOIN keep that an inner join drops?', 'left-join-anti-join'),
    ('What does PARTITION BY do inside OVER?', 'window-functions'),
    ('Which rows does the default frame cover when OVER has an ORDER BY?', 'window-functions'),
    ('How do you add a second step after the first step of a CTE?', 'ctes'),
    ('How do you debug one step of a chain of CTEs?', 'ctes'),
    ('Does date_trunc round to the nearest month?', 'dates-and-time'),
    ('What does the isodow function return?', 'dates-and-time'),
    ('Which test should come first in a CASE when two tests overlap?', 'case-expressions'),
    ('What does the ELSE branch of a CASE give?', 'case-expressions'),
]
TEST = GOLD + TEST_NEW                                                        # the 16 golden questions of lesson MLE10 and 16 new ones: scored once, never used to choose anything
print(len(TRAIN), len(DEV), len(TEST), "questions for training, validation and the test")

`search` is the retrieval of lesson MLE9: chunks of 120 words, the best ones by cosine. `Route` is the schema of lesson MLE8 with one field, and `report` gives the numbers of an approach: replies that pass the schema, right lessons, tokens, dollars for 1,000 requests at assumed prices, and median seconds. Run both cells as they are.

In [ ]:
def make_chunks(target_words):
    """Chunks of whole paragraphs, each (lesson, section, text), at most `target_words` words unless one paragraph is longer."""
    chunks = []
    for slug in LESSONS:
        current, count, head = [], 0, ""
        for section, text in CORPUS[slug]:
            size = len(text.split())
            if current and count + size > target_words:
                chunks.append((slug, head, " ".join(current)))
                current, count = [], 0
            if not current:
                head = section
            current.append(text)
            count += size
        if current:
            chunks.append((slug, head, " ".join(current)))
    return chunks


CHUNKS = make_chunks(120)
VECTORS = emb.encode([chunk[2] for chunk in CHUNKS])


def search(question, k=3):
    """The k chunks most similar to the question, as (cosine, number of the chunk)."""
    scores = VECTORS @ emb.encode([question])[0]
    return [(round(float(scores[i]), 3), int(i)) for i in np.argsort(-scores)[:k]]


print(len(CHUNKS), "chunks")

In [ ]:
class Route(BaseModel):
    lesson: Literal[tuple(LESSONS)]                                           # one of the eight names, nothing else


def checked(reply):
    """The reply with the lesson it names, or the type of its error."""
    try:
        return {**reply, "lesson": Route.model_validate_json(reply["text"]).lesson, "error": None}
    except ValidationError as problem:
        return {**reply, "lesson": None, "error": problem.errors()[0]["type"]}


PRICE_IN, PRICE_OUT = 1.00, 5.00                                           # dollars per million tokens: assumed for this lesson, not any provider's price


def report(rows, questions):
    """Replies that pass the schema, right lessons, mean prompt and completion tokens, dollars for 1,000 requests, median seconds."""
    prompt_mean = statistics.mean(r["prompt_tokens"] for r in rows)
    completion_mean = statistics.mean(r["completion_tokens"] for r in rows)
    cost = (prompt_mean * PRICE_IN + completion_mean * PRICE_OUT) / 1_000_000 * 1000
    return {"pass": sum(r["error"] is None for r in rows), "right": sum(r["lesson"] == lesson for r, (_, lesson) in zip(rows, questions)),
            "prompt": round(prompt_mean), "completion": round(completion_mean), "cost": round(cost, 3), "seconds": round(statistics.median(r["seconds"] for r in rows), 1)}

## Unit 1: Prompt it

**Predict first.** The model gets the instructions and the eight names with every question. For the 32 test questions: how many replies will pass the `Route` check, how many will name the right lesson, and how many tokens will a request count on average?

In [ ]:
SYSTEM = ("You route learner questions to lessons. Reply with JSON only, like {\"lesson\": \"ctes\"}. "
          "The value of lesson must be exactly one of: " + ", ".join(LESSONS) + ".")


def ask_prompt(question):
    """Prompting: the instructions and the eight names travel with every question."""
    return checked(chat([{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}], max_new_tokens=30))


runs = {"prompt": [ask_prompt(question) for question, _ in TEST]}
print(report(runs["prompt"], TEST))
print(sorted({r["error"] for r in runs["prompt"] if r["error"]}))

## Unit 2: Retrieve for it

**Predict first.** The same instructions, and with each question the three most similar chunks of the lessons, each starting with the name of its lesson. Will the replies name more right lessons than in unit 1, and how many tokens will a request count now? How many questions would the best chunk alone get right, with no model at all?

In [ ]:
SYSTEM_R = SYSTEM + " Sources follow the question: each one starts with the name of the lesson it comes from."


def ask_retrieval(question, k=3):
    """Retrieval: the three most similar chunks of the lessons go in with the question."""
    sources = "\n".join(f"[{CHUNKS[i][0]}] {CHUNKS[i][2]}" for _, i in search(question, k))
    return checked(chat([{"role": "system", "content": SYSTEM_R}, {"role": "user", "content": f"Question: {question}\n\nSources:\n{sources}"}], max_new_tokens=30))


runs["retrieval"] = [ask_retrieval(question) for question, _ in TEST]
print(report(runs["retrieval"], TEST))
print(sorted({r["error"] for r in runs["retrieval"] if r["error"]}), [r["text"] for r in runs["retrieval"] if r["error"]][:3])
top_chunk = [CHUNKS[search(question, 1)[0][1]][0] for question, _ in TEST]                 # no model: the lesson of the best chunk
print(sum(lesson == expected for lesson, (_, expected) in zip(top_chunk, TEST)))

## Unit 3: Tune it with LoRA

**Predict first.** The adapter wraps four kinds of layer in each of the 40 layers of the model. How many numbers will it train, out of how many in the model? Over 4 epochs, will the loss on the validation questions keep falling?

In [ ]:
SHAPES = {}                                                                   # the shape of each layer the adapter will wrap, read before wrapping
layer = llm.model.layers[0]
for name, module in [("q_proj", layer.self_attn.q_proj), ("k_proj", layer.self_attn.k_proj), ("v_proj", layer.self_attn.v_proj), ("o_proj", layer.self_attn.o_proj)]:
    SHAPES[name] = (module.in_features, module.out_features)
N_LAYERS = len(llm.model.layers)
print(SHAPES, N_LAYERS)

config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=list(SHAPES), task_type="CAUSAL_LM")
torch.manual_seed(SEED)                                                       # the first of the two small matrices starts random
llm = get_peft_model(llm, config)
trainable = sum(p.numel() for p in llm.parameters() if p.requires_grad)
total = sum(p.numel() for p in llm.parameters())
print(config.r, config.lora_alpha, config.lora_dropout, sorted(config.target_modules))
print(trainable, total, round(100 * trainable / total, 3))

The training examples and the loss: one example is the request followed by the reply, and only the reply counts for the loss. Run this cell as it is.

In [ ]:
SHORT = "Route the question to a lesson."                                    # the tuned model needs no list of names: it learns them


def example(question, lesson):
    """One training example: the token ids of the request and the reply, and which of them count for the loss (the reply only)."""
    prompt = tok.apply_chat_template([{"role": "system", "content": SHORT}, {"role": "user", "content": question}], add_generation_prompt=True, return_dict=True)["input_ids"]
    reply = tok(json.dumps({"lesson": lesson}) + "<|end_of_text|>")["input_ids"]
    return prompt + reply, [-100] * len(prompt) + reply                      # -100: no loss on the request


def batch(pairs):
    """Examples of different lengths padded to one rectangle: ids, attention mask, labels."""
    rows = [example(question, lesson) for question, lesson in pairs]
    width = max(len(ids) for ids, _ in rows)
    ids = torch.tensor([i + [tok.pad_token_id] * (width - len(i)) for i, _ in rows])
    labels = torch.tensor([lab + [-100] * (width - len(lab)) for _, lab in rows])
    mask = torch.tensor([[1] * len(i) + [0] * (width - len(i)) for i, _ in rows])
    return ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)


def mean_loss(pairs):
    """The mean loss over the reply tokens of some examples; no weight changes."""
    llm.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for start in range(0, len(pairs), 8):
            ids, mask, labels = batch(pairs[start:start + 8])
            tokens = (labels[:, 1:] != -100).sum().item()
            total += llm(input_ids=ids, attention_mask=mask, labels=labels).loss.item() * tokens
            count += tokens
    return total / count


print(len(example(*TRAIN[0])[0]), "tokens in the first example, of which", sum(label != -100 for label in example(*TRAIN[0])[1]), "are the reply")

The training loop. It takes about ten minutes on a CPU: start it and read on. It keeps the adapter of the epoch with the lowest validation loss.

In [ ]:
EPOCHS, BATCH, LEARNING_RATE = 4, 8, 5e-4
optimizer = torch.optim.AdamW([p for p in llm.parameters() if p.requires_grad], lr=LEARNING_RATE)
shuffler = torch.Generator().manual_seed(SEED)
history = [{"epoch": 0, "validation": round(mean_loss(DEV), 3)}]               # before any step
step_losses, saved = [], {}
started = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    llm.train()                                                               # dropout on while training
    order = torch.randperm(len(TRAIN), generator=shuffler).tolist()
    for first in range(0, len(TRAIN), BATCH):
        ids, mask, labels = batch([TRAIN[i] for i in order[first:first + BATCH]])
        loss = llm(input_ids=ids, attention_mask=mask, labels=labels).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        step_losses.append(round(loss.item(), 3))
    saved[epoch] = {name: p.detach().clone() for name, p in llm.named_parameters() if p.requires_grad}      # the adapter at the end of this epoch
    history.append({"epoch": epoch, "train": round(statistics.mean(step_losses[-len(TRAIN) // BATCH:]), 3), "validation": round(mean_loss(DEV), 3)})
training_seconds = round(time.perf_counter() - started)
print(len(step_losses), "steps", training_seconds, "seconds")
print(step_losses)
for row in history:
    print(row)
best = min(history[1:], key=lambda row: row["validation"])["epoch"]            # the epoch with the lowest validation loss
with torch.no_grad():
    for name, p in llm.named_parameters():
        if p.requires_grad:
            p.copy_(saved[best][name])
print("keep the adapter of epoch", best)

**Predict first.** The tuned model gets the short system message and no list of names. For the 32 test questions: how many replies will pass the check, how many will name the right lesson, and how many tokens will a request count?

In [ ]:
llm.eval()


def ask_tuned(question):
    """The tuned model: no names, no instructions, only the short system message it was trained with."""
    return checked(chat([{"role": "system", "content": SHORT}, {"role": "user", "content": question}], max_new_tokens=30))


runs["tuned"] = [ask_tuned(question) for question, _ in TEST]
print(report(runs["tuned"], TEST))
print(sorted({r["error"] for r in runs["tuned"] if r["error"]}))
wrong = [i for i, (r, (_, lesson)) in enumerate(zip(runs["tuned"], TEST)) if r["lesson"] != lesson]
for i in wrong:
    print(i, TEST[i][1], "->", runs["tuned"][i]["text"])

## Unit 4: What the adapter is

**Predict first.** The adapter is saved to a file and the same question is sent twice, once with the adapter switched off and once with it on. How large is the file, and what will the tuned model answer to a question that is not a routing request?

In [ ]:
ADAPTER_DIR = tempfile.mkdtemp()
llm.save_pretrained(ADAPTER_DIR)
print(round(os.path.getsize(ADAPTER_DIR + "/adapter_model.safetensors") / 1e6, 1), "MB in the adapter file")
question = [{"role": "user", "content": "In one sentence, what does COALESCE do in SQL?"}]
with llm.disable_adapter():                                                   # the same model, the adapter switched off
    print(repr(chat(question, max_new_tokens=40)["text"]))
print(repr(chat(question, max_new_tokens=40)["text"]))                       # the adapter on

## Unit 5: The bill

**Predict first.** The prices are the assumed ones of lesson MLE8. Labeling the 112 questions is given an invented cost of $200. After how many requests has the shorter prompt repaid it, on how many questions do the tuned model and retrieval differ, and is the gap between them more than chance?

In [ ]:
reports = {name: report(rows, TEST) for name, rows in runs.items()}
for name, row in reports.items():
    print(name, row)
LABELING = 200.00                                                             # dollars to write and check 112 labeled questions: invented for this lesson
saved = (reports["prompt"]["cost"] - reports["tuned"]["cost"]) / 1000       # dollars saved on one request by the shorter prompt
print(round(saved, 6), round(LABELING / saved))                              # requests until the shorter prompt has repaid the labeling
print(round(LABELING / saved / 400_000, 1))                                  # months at 400,000 requests a month, the invented volume of lesson MLE8
right = {name: [r["lesson"] == lesson for r, (_, lesson) in zip(rows, TEST)] for name, rows in runs.items()}


def differ(a, b):
    """The questions that a gets right and b gets wrong, and the questions that b gets right and a gets wrong."""
    return sum(x and not y for x, y in zip(right[a], right[b])), sum(y and not x for x, y in zip(right[a], right[b]))


print(differ("tuned", "retrieval"), differ("tuned", "prompt"), differ("retrieval", "prompt"))


def sign_test(a, b):
    """The exact two-sided sign test: if the two approaches were equally good, how often would the questions where they differ split at least this unevenly?"""
    n, k = a + b, min(a, b)
    return round(min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n), 3)


print(sign_test(*differ("tuned", "retrieval")), sign_test(*differ("tuned", "prompt")), sign_test(*differ("retrieval", "prompt")))

## Finish it

### Exercise 1: Finish it

**Task:** **Count the numbers an adapter trains** The adapter of unit 3 has rank 8 and wraps the four attention layers. Suppose it had rank 16 and wrapped the two layers of the feed-forward block as well: `input_linear` maps 2,048 numbers to 8,192, and `output_linear` maps 4,096 back to 2,048. Put in `answer_1` how many numbers the adapter would train in all 40 layers of the model. A layer of shape (n_in, n_out) gets two small matrices, `R x n_in` and `n_out x R`.

**Expected:** `answer_1` is a whole number above 10 million.

In [ ]:
R = 16
shapes = {**SHAPES, "input_linear": (2048, 8192), "output_linear": (4096, 2048)}                          # the four attention layers of unit 3 and the two layers of the feed-forward block
per_layer = sum(R * (n_in + n_out) for n_in, n_out in shapes.values())                                    # given: the numbers trained in one transformer layer
answer_1 = None  # replace None with the numbers trained in all the layers

In [ ]:
_ = check("Exercise 1", "answer_1", 19005440, traps={475136: "That is one layer: the model has 40."})

<details>
<summary>Hint</summary>

Each wrapped layer adds `R * n_in + n_out * R` numbers. `per_layer` already adds that up for the six layers of one transformer layer.

</details>

<details>
<summary>Half-solution</summary>

```python
per_layer = sum(R * (n_in + n_out) for n_in, n_out in shapes.values())
answer_1 = per_layer * ...
```

</details>

<details>
<summary>Solution</summary>

```python
R = 16
shapes = {**SHAPES, "input_linear": (2048, 8192), "output_linear": (4096, 2048)}
per_layer = sum(R * (n_in + n_out) for n_in, n_out in shapes.values())
answer_1 = per_layer * N_LAYERS
```

The answer is 19005440. One layer trains 16 x 13,312 = 212,992 numbers in the attention layers and 16 x 16,384 = 262,144 in the feed-forward block, 475,136 in all; 40 layers give 19,005,440, about 1.2 percent of the model. The rank and the number of wrapped layers set the size.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Which of the three?** Before you start, say in one sentence which of prompt, retrieve and fine-tune you expect for each problem. All three are invented. (a) A cafe chain wants replies about its loyalty rewards, which change on the first of every month; it has no labeled replies. (b) A support team has 40,000 replies in its house style and sends 2 million requests a month; a prompt with the style guide and six examples carries 900 tokens and is accepted for 70 percent of requests, and the team needs 90 percent. (c) An intern needs a one-off summary of 30 customer notes in a format given today, with no labeled data. Then write the cell yourself, in the empty cell below: put in `answer_2` a list of three words, one for each problem, each one of `'prompt'`, `'retrieve'` or `'fine-tune'`.

**Expected:** `answer_2` is a list of three words.

In [ ]:
_ = check("Exercise 2", "answer_2", ["retrieve", "fine-tune", "prompt"])

<details>
<summary>Hint</summary>

Ask of each problem what has to change: the facts the model can see, the behavior the model shows, or only the instruction. Then ask how often the thing changes and how many labeled examples exist.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_2 = [..., ..., ...]
```

</details>

<details>
<summary>Solution</summary>

```python
answer_2 = ["retrieve", "fine-tune", "prompt"]
```

The answer is retrieve, fine-tune, prompt. In (a) the facts change every month and nothing is labeled: retrieved text is new the day it is written, and a tuned model would hold last month's rewards. In (b) the behavior is the problem (70 percent against 90), 40,000 examples exist, and each request carries 900 tokens that a tuned model would not need. In (c) one instruction is enough for a one-off job.

</details>

## Check yourself

### Exercise 3: Check yourself

**Task:** **Spot the flaw** A colleague builds a test set from 35 candidate questions and keeps those that are not in the training file, comparing the exact text. Change the code so that it counts the candidates that are really not in the training file, and put the number in `answer_3`.

**Expected:** `answer_3` is a whole number.

In [ ]:
pool = [question for question, _ in TEST] + [TRAIN[0][0].lower().rstrip("?"), TRAIN[7][0].upper(), TRAIN[20][0] + " "]      # a colleague's 35 candidates for a new test set
trained_on = {question for question, _ in TRAIN}
held_out = [question for question in pool if question not in trained_on]                                                  # colleague: "not in the training file"
answer_3 = len(held_out)
# change the code so that it counts the candidates that are really not in the training file

In [ ]:
_ = check("Exercise 3", "answer_3", 32, traps={35: "The exact text misses the three questions that differ only in case, punctuation or spaces."})

<details>
<summary>Hint</summary>

Three of the candidates are training questions with a small change. Compare the words, not the exact text.

</details>

<details>
<summary>Half-solution</summary>

```python
def plain(text):
    return ...


trained_on = {plain(question) for question, _ in TRAIN}
```

</details>

<details>
<summary>Solution</summary>

```python
pool = [question for question, _ in TEST] + [TRAIN[0][0].lower().rstrip("?"), TRAIN[7][0].upper(), TRAIN[20][0] + " "]


def plain(text):
    """The words of a question in lower case, without punctuation or extra spaces."""
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", text.lower()).split())


trained_on = {plain(question) for question, _ in TRAIN}
held_out = [question for question in pool if plain(question) not in trained_on]
answer_3 = len(held_out)
```

The answer is 32. Three candidates are training questions in lower case without the question mark, in capitals, and with a trailing space; the exact comparison keeps them, so the model would be scored on questions it trained on, which shows what it remembers. Comparing the words after lower-casing and removing punctuation keeps only the 32 questions of `TEST`. A paraphrase is not caught by this: compare embeddings for that.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")